### Ingestão de dados e tabela Bronze

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS delivery;

In [0]:
# dbfs:/FileStore/delivery/

In [0]:
df = (
    spark.read
    .option("header","true")
    .option("inferSchema","true")
    .csv("/Volumes/workspace/delivery/delivery_data/online_food_delivery_dataset.csv")
)

In [0]:
from pyspark.sql.functions import col

# Sanitize column names: replace spaces with underscores
for old_name in df.columns:
    new_name = old_name.replace(" ", "_")
    if old_name != new_name:
        df = df.withColumnRenamed(old_name, new_name)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("delivery.bronze_online_delivery")

In [0]:
display(spark.table("delivery.bronze_online_delivery").limit(20))

### **Tatamento de qualidade - Camada Silver**

In [0]:
from pyspark.sql.functions import col

df = spark.table("delivery.bronze_online_delivery")

for c in df.columns:
    df = df.withColumnRenamed(
        c,
        c.strip()
         .replace(" ","_")
         .replace(".","")
         .lower()
    )

In [0]:
df = df.drop("_c13")

Criar Labels

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "label_output",
    when(col("output")=="Yes",1).otherwise(0)
)

In [0]:
df = df.withColumn(
    "label_feedback",
    when(col("feedback").contains("Positive"),1)
    .otherwise(0)
)
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("delivery.silver_online_delivery")

In [0]:
display(spark.table("delivery.silver_online_delivery").limit(20))

In [0]:
df = spark.table("delivery.silver_online_delivery")

# Descreve as colunas do DataFrame atual
df.printSchema()

# Estatísticas resumidas para todas as colunas
display(df.summary())

# Nomes de coluna e tipos de dados
for col_name, dtype in df.dtypes:
    print(f"{col_name:<40} {dtype}")

**ENGENHARIA DE FEATURES**

In [0]:
# Faixa etária
df = df.withColumn(
    "age_group",
    when(col("age") < 25,"young")
    .when(col("age") < 30,"adult")
    .otherwise("senior")
)

In [0]:
# Tamanho da família
df = df.withColumn(
    "big_family",
    when(col("family_size") >= 5,1)
    .otherwise(0)
)

In [0]:
# Cliente recorrente
df = df.withColumn(
    "is_loyal",
    when(col("customer_type")=="Frequent",1)
    .otherwise(0)
)

In [0]:
display(df.limit(20))

Cluster Geográfico

In [0]:
from pyspark.ml.clustering import KMeans

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

# Assemble geo features into a single vector column
assembler = VectorAssembler(
    inputCols=["latitude", "longitude", "pin_code"],
    outputCol="geo_features"
)
df = assembler.transform(df)

# Train KMeans clustering model (k=5 as default)
kmeans = KMeans(k=5, seed=42, featuresCol="geo_features", predictionCol="region_cluster")
model = kmeans.fit(df)

# Add cluster prediction column
df = model.transform(df)

# Drop the intermediate vector column
df = df.drop("geo_features")

In [0]:
display(df.groupBy("region_cluster").count().orderBy("region_cluster"))

In [0]:
features = [
    "age",
    "family_size",
    "latitude",
    "longitude",
    "region_cluster"
]

In [0]:
categorical_columns = [
    "gender",
    "marital_status",
    "occupation",
    "monthly_income",
    "educational_qualifications",
    "customer_type",
    "age_group"
]

In [0]:
numeric_cols = [
    "age",
    "family_size",
    "big_family",
    "is_loyal"
]

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("delivery.gold_consumer_features")

In [0]:
display(spark.table("delivery.gold_consumer_features").limit(20))

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

df = spark.table("delivery.gold_consumer_features")

# Indexa e aplica one-hot encoding em colunas categóricas

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_columns
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_columns],
    outputCols=[f"{c}_ohe" for c in categorical_columns]
)

# Combina todos os recursos em uma única coluna de vetor.


assembler = VectorAssembler(
    inputCols=
        numeric_cols +
        [f"{c}_ohe" for c in categorical_columns],
    outputCol="features"
)

# Remove quaisquer colunas intermediárias criadas anteriormente.

for c in df.columns:
    if c.endswith("_idx") or c.endswith("_ohe") or c.endswith("_vec") or c == "features":
        df = df.drop(c)

# Constroi e ajusta o pipeline de pré-processamento.

pipeline_features = Pipeline(
    stages=
        indexers +
        [encoder, assembler]
)
feature_model = pipeline_features.fit(df)

df_features = feature_model.transform(df)

In [0]:
display(
    df_features.select(
        "features"
    )
)

In [0]:
dataset_output = df_features.select(
    "features",
    "label_output"
)

In [0]:
dataset_feedback = df_features.select(
    "features",
    "label_feedback"
)

In [0]:
# Modelo de output para treinamento 80/20

train_feedback, test_feedback = (
    dataset_feedback.randomSplit(
        [0.8, 0.2],
        seed=42
    )
)

In [0]:
train_output, test_output = (
    dataset_output.randomSplit(
        [0.8, 0.2],
        seed=42
    )
)

print("Treino:", train_output.count())
print("Teste:", test_output.count())

In [0]:
#Treinamento do modelo de output
from pyspark.ml.classification import RandomForestClassifier

rf_output = RandomForestClassifier(
    labelCol="label_output",
    featuresCol="features",
    numTrees=300,
    maxDepth=8,
    seed=42
)

model_output = rf_output.fit(train_output)

In [0]:
# Treinamento do modelo de feedback
rf_feedback = RandomForestClassifier(
    labelCol="label_feedback",
    featuresCol="features",
    numTrees=300,
    maxDepth=8,
    seed=42
)

model_feedback = rf_feedback.fit(train_feedback)

In [0]:
# Fazer Previsões
pred_output = model_output.transform(test_output)

pred_feedback = model_feedback.transform(test_feedback)

In [0]:
display(pred_output.limit(20))
display(pred_feedback.limit(20))

In [0]:
# Calcular AUC para avaliar performance do modelo
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label_output",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(pred_output)

print(f"AUC: {auc:.4f}")

In [0]:
importances = model_output.featureImportances
feature_names = assembler.getInputCols()

for f, imp in zip(feature_names, importances):
    print(f, imp)


In [0]:
# Ranking de influência dos atributos

importance_df = spark.createDataFrame(
    zip(feature_names,
        [float(x) for x in importances]),
    ["feature", "importance"]
)

display(
    importance_df.orderBy(
        importance_df.importance.desc()
    )
)